# Phase 3.2 · Where does within-KV-group specialization originate?

**Observational only.** This notebook does not load the model, modify weights,
patch activations, or ablate anything — those are Phase 3.3. It reads the
artefacts Phase 3.1 already saved and decomposes the attention computation to
localise the stage at which heads sharing a KV head begin to differ.

Phase 3.1 established (Part 12, the ICC) that heads in the same KV group are more
alike than heads across groups, but a substantial within-group difference remains
and **grows with depth**. Since within a group the Keys and Values are identical,

$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\tfrac{QK^\top}{\sqrt d}\right)V,$$

any divergence must enter through the **query path**. The four analyses below
walk that path: query magnitude → raw $QK^\top$ → BOS competition → drift through
depth.

## Part 0 · Setup

Mount Drive, locate `phase32_utils.py` (and its dependency `phase3_utils.py`),
and point `EXP_DIR` at the Phase 3.1 experiment whose artefacts we reuse.

In [ ]:
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/attention_sink_project')
else:
    BASE = Path('.')

# discover the helper modules (both are needed: phase32 imports phase3)
for cand in [BASE, Path('/content'), Path('.')]:
    if (cand / 'phase32_utils.py').exists():
        sys.path.insert(0, str(cand)); break
for cand in [BASE, Path('/content'), Path('.')]:
    if (cand / 'phase3_utils.py').exists():
        sys.path.insert(0, str(cand)); break

import phase3_utils as U
import phase32_utils as P2
import numpy as np, torch
print('phase3_utils + phase32_utils loaded — no model will be loaded in this notebook')

# Phase 3.1 experiment directory (its captures/metrics/analysis feed Phase 3.2)
EXP_DIR = BASE / 'results' / 'phase3' / 'experiment3.1'
OUT_DIR = BASE / 'results' / 'phase3' / 'phase3_2_origin'
assert (EXP_DIR / 'metrics' / 'table.pt').exists(), f'Phase 3.1 outputs not found under {EXP_DIR}'
print('reading Phase 3.1 artefacts from:', EXP_DIR)
print('writing Phase 3.2 outputs to    :', OUT_DIR)

In [ ]:
# --- Load everything once (reuses the Phase 3.1 loaders; no inference) -------
inp = P2.load_phase32_inputs(EXP_DIR)
print(f'layers={inp.n_layers}  heads={inp.n_heads}  KV groups={inp.n_groups}  n_rep={inp.n_rep}')
print(f'captures loaded: {len(inp.captures)}   within-group head pairs: {len(inp.head_pairs())}')

# per-head aggregates (feeds Analyses 1 & 3 and the CSV) — from saved scalars only
head_tbl = P2.build_head_table(inp)
csv_path = P2.write_summary_csv(head_tbl, OUT_DIR / 'summary.csv')
print('wrote', csv_path, f'({len(head_tbl.layer)} rows = layers × heads)')

## Analysis 1 · Query vector statistics

**Motivation.** The simplest way two shared-KV heads could differ is by emitting
query vectors of different *magnitude*. **Computation.** Per head, the L2 norm
$\|q\|_2$ (post-RoPE, post-QK-norm) averaged over tokens, plus the within-group
spread of those means. **Expected interpretation.** Qwen3 applies QK-RMSNorm
before RoPE (both norm-preserving up to the learned $\gamma$), so $\|q\|$ lives in
a band shared by every head. If the within-group spread is a small fraction of
that band, magnitude is *not* the source of specialization and we must look at
direction (Analysis 2).

In [ ]:
qns = P2.query_norm_stats(inp, head_tbl)
print(f'query-norm band across all heads/layers: '
      f'[{qns.global_min:.3f}, {qns.global_max:.3f}]')
print(f'mean within-group spread of head-mean norms: '
      f'{np.nanmean(qns.within_group_spread):.4f}')
fig1 = P2.plot_query_norms(qns, OUT_DIR / 'figures' / 'analysis1_query_norms.png', show=True)

## Analysis 2 · Raw compatibility scores ($QK^\top$, before masking/softmax)

**Motivation.** If magnitude is equal, do the heads' *raw* compatibility patterns
already differ? **Computation.** Recompute $QK^\top$ from the stored $q,k$ (K is
shared within a group, so any difference isolates the query projection). We show
heatmaps for a shared-KV pair at an early, middle, and late layer, their
difference, and the Pearson correlation of the causal $QK^\top$ entries between
paired heads across depth. **Expected interpretation.** Low within-group
correlation means specialization is present *before* softmax — it originates in
the query projection, not in the softmax competition.

In [ ]:
# Recompute is done one layer at a time inside the helper (memory ~[H,T,T]).
# Shared with Analysis 4, so we compute the similarity core once here.
sim = P2.within_group_similarity(inp)
print('mean within-group correlation of raw QK^T: '
      f'{np.nanmean(sim.score_corr):.3f}  (1 = identical pattern, 0 = unrelated)')
fig2 = P2.plot_compatibility(inp, sim, OUT_DIR / 'figures' / 'analysis2_compatibility.png',
                             group=0, show=True)

## Analysis 3 · BOS competition

**Motivation.** A sink head could earn its BOS attention two ways: by *favouring
BOS* (a larger $q\cdot k_{\text{BOS}}$) or by *suppressing competitors* (a smaller
largest-competitor logit). **Computation.** Per head, the BOS advantage
$= \text{BOS logit} - \max_i(\text{competitor logit})$ — exactly Phase 3.1's
`margin_max`. Heads are split into sink / non-sink by their sink score (mean BOS
probability), and we compare the two logit terms between the groups. **Expected
interpretation.** Whichever term moves more between sink and non-sink heads is the
mechanism the sink relies on.

In [ ]:
bc = P2.bos_competition(inp, head_tbl, sink_quantile=0.75)
print(f'sink heads (top 25% by sink score): {bc.sink_heads}')
print(f'  mean BOS advantage   sink {np.nanmean(bc.adv_sink):+.3f}  '
      f'vs non-sink {np.nanmean(bc.adv_nonsink):+.3f} nats')
print(f'  mean BOS logit       sink {bc.bos_logit_sink_mean:+.3f}  '
      f'vs non-sink {bc.bos_logit_nonsink_mean:+.3f}')
print(f'  mean top competitor  sink {bc.competitor_sink_mean:+.3f}  '
      f'vs non-sink {bc.competitor_nonsink_mean:+.3f}')
fig3 = P2.plot_bos_competition(bc, OUT_DIR / 'figures' / 'analysis3_bos_competition.png', show=True)

## Analysis 4 · Divergence through depth

**Motivation.** The key mechanistic question: do shared-KV heads start (nearly)
identical and drift apart, or are they already specialized in the earliest layers?
**Computation.** For each KV group and layer, three within-group similarities —
cosine of query vectors, correlation of raw $QK^\top$, correlation of BOS logits —
averaged over sequences (the core computed in Analysis 2 is reused). **Expected
interpretation.** A monotone decline with depth means specialization develops
during representation refinement; a flat-low profile means it is present from the
start. This localises, in depth, the within-group variation Phase 3.1's ICC only
summarised.

In [ ]:
fig4 = P2.plot_divergence(sim, OUT_DIR / 'figures' / 'analysis4_divergence.png', show=True)
for name, arr in (('query cosine', sim.query_cosine), ('QK^T corr', sim.score_corr),
                  ('BOS-logit corr', sim.bos_logit_corr)):
    print(f'  {name:<16} first layer {np.nanmean(arr[0]):+.3f}  ->  '
          f'last layer {np.nanmean(arr[-1]):+.3f}')

## Interpretation

The statements below are generated from the measurements in this run (see
`interpret`), not hard-coded. They are the Phase 3.2 findings.

In [ ]:
obs = P2.interpret(qns, sim, bc)
import json
for k in ('analysis1', 'analysis2', 'analysis3', 'analysis4'):
    print(f'\n[{k}]\n{obs[k]}')
(OUT_DIR / 'observations.json').write_text(json.dumps(obs, indent=2))

# persist intermediate tensors for Phase 3.3
(OUT_DIR / 'tensors').mkdir(parents=True, exist_ok=True)
torch.save({'per_head_mean': qns.per_head_mean, 'within_group_spread': qns.within_group_spread,
            'layers': qns.layers}, OUT_DIR / 'tensors' / 'query_norm_stats.pt')
torch.save({'pairs': sim.pairs, 'pair_groups': sim.pair_groups, 'layers': sim.layers,
            'query_cosine': sim.query_cosine, 'score_corr': sim.score_corr,
            'bos_logit_corr': sim.bos_logit_corr}, OUT_DIR / 'tensors' / 'within_group_similarity.pt')
print('\nsaved observations.json and intermediate tensors')

## Part N · Download

Bundle the figures, `summary.csv`, observations, and intermediate tensors.

In [ ]:
import shutil, tempfile
stage = Path(tempfile.mkdtemp()) / 'phase3_2_origin'
shutil.copytree(OUT_DIR, stage)
zp = shutil.make_archive(str(Path('/content' if IN_COLAB else '.') / 'phase3_2_origin'), 'zip', stage)
print('bundled:', zp, f'({Path(zp).stat().st_size/1024:.0f} KB)')
if IN_COLAB:
    from google.colab import files
    files.download(zp)